# Gaussian Boson Sampling (GBS)

Jupyter para aprender a usar o algoritmo **GBS** encontrado na url [Sampling from GBS (StrawberryFields)](https://strawberryfields.ai/photonics/apps/run_tutorial_sample.html) 

## Sampling from GBS

O dispositivo **GBS** pode ser programado para amostrar de qualquer matriz simétrica $A$.

Para amostrar, devemos especificar o número médio de fótons gerados no dispositivo e, opcionalmente, o método de detecção usado na saída: detecção por limiar ou detecção por resolução do número de fótons (*Photons**PNR**). Detectores de limiar (*Threshold detectors*) se restringem a medir se os fótons chegaram ao detector, enquanto detectores PNR conseguem contar o número de fótons. A perda de fótons também pode ser especificada com o argumento `loss`.

A funcionalidade de amostragem é fornecida no módulo `sample`.

Vamos analisar os dois tipos de métodos de amostragem. Podemos gerar amostras a partir de uma matriz simétrica aleatória de 5 dimensões:

In [4]:
from strawberryfields.apps import sample
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

modes = 5
n_mean = 6
samples = 5

A = np.random.normal(0, 1, (modes, modes))
A = A + A.T # A.T -> transposta da matriz A

s_thresh = sample.sample(A, n_mean, samples, threshold=True)
s_pnr = sample.sample(A, n_mean, samples, threshold=False)

print(np.array(s_thresh))
print(np.array(s_pnr))

[[0 0 0 0 0]
 [1 0 0 1 0]
 [0 0 0 0 0]
 [1 0 1 1 0]
 [1 0 1 1 0]]
[[3 0 0 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [2 0 2 0 0]]



Em cada caso, uma amostra é uma sequência de inteiros de comprimento cinco, ou seja, ```len(modos) = 5```. As amostras de limiar são $0$s e $1$s, correspondendo à detecção ou não de fótons em um determinado modo. Um $1$ aqui é convencionalmente chamado de "*clique*". As amostras PNR são inteiros não negativos que contam o número de fótons detectados em cada modo. Por exemplo, suponha que uma amostra PNR seja $[2, 1, 1, 0, 0]$, o que significa que $2$ fótons foram detectados no modo $0$, $1$ fóton foi detectado nos modos $1$ e $2$ e $0$ fótons foram detectados nos modos $3$ e $4$. Se detectores de limiar fossem usados, a amostra seria: $[1, 1, 1, 0, 0]$.

Uma função ```gaussiana()``` mais geral permite a amostragem de estados gaussianos puros arbitrários.

### Amostragem de subgrafos

Então, quando a detecção por limiar ou a detecção por PNR seria preferível em GBS? Como as amostras de limiar podem ser pós-processadas a partir de amostras de PNR, poderíamos esperar que a detecção por PNR fosse sempre a escolha preferida. No entanto, na prática, a simulação de GBS baseada em PNR é significativamente mais lenta e verifica-se que as amostras de limiar podem fornecer informações úteis suficientes para uma variedade de aplicações.

O Strawberry Fields fornece ferramentas para resolver problemas baseados em grafos. Nesse contexto, normalmente queremos usar GBS para amostrar subgrafos, que provavelmente serão densos devido à distribuição de probabilidade do GBS [1]. 
>[1]
>Juan Miguel Arrazola and Thomas R Bromley. Using gaussian boson sampling to find dense subgraphs. Physical Review Letters, 121(3):030503, 2018.


Nesse caso, a amostragem por limiar é suficiente, pois nos permite selecionar nós do subgrafo. Vejamos isso usando um pequeno grafo fixo como exemplo:

In [5]:
from strawberryfields.apps import plot
import networkx as nx
import plotly

adj = np.array(
    [
        [0, 1, 0, 0, 1, 1],
        [1, 0, 1, 0, 1, 1],
        [0, 1, 0, 1, 1, 0],
        [0, 0, 1, 0, 1, 0],
        [1, 1, 1, 1, 0, 1],
        [1, 1, 0, 0, 1, 0],
    ]
)

graph = nx.Graph(adj)
plot.graph(graph)


Este é um grafo de 6 nós, com os nós $[0, 1, 4, 5]$ totalmente conectados entre si. Esperamos poder amostrar subgrafos densos com alta probabilidade.

As amostras podem ser geradas a partir deste grafo por meio do GBS usando a função `sample()`:

In [6]:
n_mean = 4
samples = 20

s = sample.sample(adj, n_mean, samples)

print(np.array(s[:5]))

[[0 0 0 0 0 0]
 [0 1 1 0 1 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [1 0 1 1 1 1]]


Cada amostra em `s` é uma lista de modos, com 1 para nós que clicaram e 0 para nós que não clicaram. Queremos converter uma amostra para outra representação, onde o resultado seja uma lista de modos que indicaram que houve cliques. Essa lista de modos pode ser usada para selecionar um subgrafo. Por exemplo, se $[0, 1, 0, 1, 1, 0]$ for uma amostra de GBS, então $[1, 3, 4]$ são os nós selecionados do subgrafo correspondente.

No entanto, o número de cliques em GBS é uma variável aleatória e nem sempre há garantia de que uma amostra terá cliques suficientes para que o subgrafo resultante seja de interesse. Podemos filtrar as amostras irrelevantes usando a função `postselect()`.

In [7]:
min_clicks = 3
max_clicks = 4

s = sample.postselect(s, min_clicks, max_clicks)

print(len(s))
s.append([0, 1, 0, 1, 1, 0])


6


Como esperado, temos menos amostras do que antes. O número de amostras que sobrevivem a essa pós-seleção é determinado pelo número médio de fótons no GBS. Também adicionamos à nossa amostra de exemplo $[0, 1, 0, 1, 1, 0]$ para garantir que haja pelo menos uma para o seguinte.

Vamos converter nossas amostras pós-selecionadas em subgrafos:

In [8]:
subgraphs = sample.to_subgraphs(s, graph)

print(subgraphs)


[[1, 2, 4], [0, 1, 2], [0, 1, 4, 5], [0, 1, 2, 4], [1, 2, 4], [2, 3, 4], [1, 3, 4]]



Podemos analisar um dos subgrafos amostrados:

In [9]:
plot.graph(graph, subgraphs[0])


Esses subgrafos amostrados servem como ponto de partida para algumas das aplicações disponibilizadas no Strawberry Fields, incluindo os problemas de clique máxima e de identificação de subgrafos densos.

>[Nota]
>
> A simulação de GBS pode ser computacionalmente intensiva ao usar detectores de limiar e PNR. Afinal, estamos usando um algoritmo clássico para simular um processo quântico! Para ajudar os usuários a se familiarizarem com as aplicações do Strawberry Fields o mais rápido possível, fornecemos conjuntos de dados de amostras de GBS pré-calculadas. Esses conjuntos de dados estão disponíveis no módulo de dados.